# 06 | Research Efficiency and Returns to Scale

This notebook quantifies how much research output countries get per dollar spent
and tests whether returns to scale are increasing, constant, or diminishing.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
panel = pd.read_csv("../data/processed/merged_panel.csv")


## Publications per million USD, by technology and year

In [ ]:
eff = (
    panel.groupby(["technology", "year"], as_index=False)
         .agg(spend=("spending_usd_ppp_millions", "sum"),
              pubs=("pub_count", "sum"))
)
eff["pubs_per_M"] = eff["pubs"] / eff["spend"].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(11, 6))
for tech, sub in eff.groupby("technology"):
    ax.plot(sub["year"], sub["pubs_per_M"], label=tech, lw=1.5)
ax.set_yscale("log")
ax.set_xlabel("Year")
ax.set_ylabel("Publications per USD M (log scale)")
ax.set_title("Research efficiency by technology")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=9)
plt.tight_layout()
plt.savefig("../figures/efficiency_trends.png", dpi=150, bbox_inches="tight")
plt.show()


## Country efficiency heatmap

In [ ]:
country_eff = (
    panel.groupby(["country", "technology"], as_index=False)
         .agg(spend=("spending_usd_ppp_millions", "sum"),
              pubs=("pub_count", "sum"))
)
country_eff["pubs_per_M"] = country_eff["pubs"] / country_eff["spend"].replace(0, np.nan)
heat = country_eff.pivot(index="country", columns="technology", values="pubs_per_M")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(np.log10(heat.fillna(0) + 0.1), annot=False, cmap="viridis", ax=ax,
            cbar_kws={"label": "log10(pubs per USD M + 0.1)"})
ax.set_title("Research efficiency: log10(pubs / USD M) by country × technology")
plt.tight_layout()
plt.savefig("../figures/efficiency_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## Log-log elasticity regression: log(pubs) ~ log(spending) + country FE + year FE

In [ ]:
sub = panel.query("spending_usd_ppp_millions > 0 and pub_count > 0").copy()
sub["log_spend"] = np.log(sub["spending_usd_ppp_millions"])
sub["log_pubs"]  = np.log(sub["pub_count"])

results = []
for tech in sorted(sub["technology"].unique()):
    df = sub.query("technology == @tech")
    if len(df) < 100:
        continue
    X = df[["log_spend"]].copy()
    X = X.join(pd.get_dummies(df["country"], drop_first=True).astype(float))
    X = X.join(pd.get_dummies(df["year"], drop_first=True, prefix="y").astype(float))
    X = sm.add_constant(X)
    m = sm.OLS(df["log_pubs"], X).fit(cov_type="HC1")
    results.append({
        "technology": tech,
        "beta": m.params["log_spend"],
        "se": m.bse["log_spend"],
        "n": int(m.nobs),
        "r2": m.rsquared,
    })

elast = pd.DataFrame(results).sort_values("beta")
elast.to_csv("../results/efficiency_and_scale.csv", index=False)
elast


## Forest plot of elasticities

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(elast["beta"], elast["technology"], xerr=1.96 * elast["se"],
            fmt="o", color="black", capsize=3)
ax.axvline(0, color="grey", ls="--", lw=0.8)
ax.axvline(1, color="red", ls="--", lw=0.8, label="constant returns")
ax.set_xlabel("β: elasticity of log(pubs) w.r.t. log(spending)")
ax.set_title("Returns to scale by technology (country + year FE)")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/coefficient_plot_grid.png", dpi=150, bbox_inches="tight")
plt.show()


## Hydrogen scatter, illustrating the relationship

In [ ]:
h = sub.query("technology == 'Hydrogen & fuel cells'")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(h["log_spend"], h["log_pubs"], alpha=0.4, s=15)
b = elast.set_index("technology").loc["Hydrogen & fuel cells", "beta"]
xs = np.linspace(h["log_spend"].min(), h["log_spend"].max(), 100)
ax.plot(xs, h["log_pubs"].mean() + b * (xs - h["log_spend"].mean()),
        color="red", lw=2, label=f"β = {b:.2f}")
ax.set_xlabel("log(spending USD M)")
ax.set_ylabel("log(publications)")
ax.set_title("Hydrogen & fuel cells, log-log scaling")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/hydrogen_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# Returns-to-scale summary plot
fig, ax = plt.subplots(figsize=(7, 4))
elast.set_index("technology")["beta"].sort_values().plot.barh(ax=ax, color="steelblue")
ax.axvline(1, color="red", ls="--", lw=0.8)
ax.set_xlabel("β (1.0 = constant returns)")
plt.tight_layout()
plt.savefig("../figures/returns_to_scale.png", dpi=150, bbox_inches="tight")
plt.show()


## Output

- `../results/efficiency_and_scale.csv`, per-technology elasticities and standard errors.
- `../figures/efficiency_trends.png`, `efficiency_heatmap.png`,
  `coefficient_plot_grid.png`, `hydrogen_scatter.png`, `returns_to_scale.png`.

**Findings:** Hydropower yields ~24 pubs/M$, nuclear only ~0.4, a 60× efficiency
gap. All elasticities lie below 1.0, indicating diminishing returns to scale
within country-year panels.
